# Phase 5: seeds, lead II, CODE-15% external validation

Run on Google Colab with Google Drive mounted. Each phase starts only if the previous phase's gate file says GO. See the repository README for the full workflow.

### Cell 5.0: Start the session and shared helpers

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, sys, json, shutil, importlib, time, math, hashlib, subprocess, zipfile, datetime
from pathlib import Path
import numpy as np, pandas as pd, torch

PROJECT = Path("/content/drive/MyDrive/ecg_leak")
sys.path.insert(0, str(PROJECT / "src"))
import ecgleak.config as ecfg, ecgleak.models as M, ecgleak.train as TR, ecgleak.evaluate as EV, \
    ecgleak.leakage as LK, ecgleak.analysis as AN
for mod in (ecfg, M, TR, EV, LK, AN):
    importlib.reload(mod)
CFG = ecfg.load(str(PROJECT / "config.yaml"))
PATHS = CFG["paths"]
gate3 = json.load(open(f"{PATHS['gates']}/gate_3.json"))
gate4 = json.load(open(f"{PATHS['gates']}/gate_4.json"))
PLAN = json.load(open(f"{PATHS['gates']}/phase3_plan.json"))
assert gate4["go"], "Phase 4 gate is not GO"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WINDOW = int(CFG["preprocess"]["window_s"] * CFG["data"]["fs"])
OUT = f"{PATHS['results']}/phase5"
os.makedirs(OUT, exist_ok=True)
PRIMARY = CFG["leakage"]["primary"]

LOCAL_CACHE = "/content/cache"
os.makedirs(LOCAL_CACHE, exist_ok=True)
splits = pd.read_csv(f"{PATHS['splits']}/splits.csv", index_col="ecg_id")
def load_npz(src, dates=True):
    dst = f"{LOCAL_CACHE}/{os.path.basename(os.path.dirname(src))}_{os.path.basename(src)}"
    if not os.path.isfile(dst):
        shutil.copyfile(src, dst)
    z = np.load(dst, allow_pickle=False)
    d = {k: z[k] for k in z.files}
    if dates:
        d["date"] = pd.to_datetime(splits.loc[d["ecg_id"], "recording_date"]).to_numpy()
    return d
D = {s: load_npz(f"{PATHS['cache']}/{s}.npz") for s in ["id_train", "id_eval", "probe_dev", "lockbox"]}
SUPER = list(D["probe_dev"]["label_names"])
UNTRAINED = M.untrained_copies(CFG, PLAN["untrained_seeds"], DEVICE)

def leak_gains(enc, dev, test, task="CD"):
    """Phase 3 protocol: attacker fit on dev (5 patient folds), scored once on test; three gains with CIs."""
    keep_d, y_d = EV.task_data(dev["labels"], SUPER, task)
    keep_t, y_t = EV.task_data(test["labels"], SUPER, task)
    def rep(d, keep):
        sig = d["signal"][keep]
        r = {"identity_template": EV.embed(enc, sig, WINDOW, DEVICE)}
        for k, m in enumerate(UNTRAINED):
            r[f"untrained_{k}"] = EV.embed(m, sig, WINDOW, DEVICE)
        r["raw_median_beat"] = d["median_beat"][keep]
        r["age_sex"] = EV.demo_features(d["age"][keep], d["sex"][keep])
        r["template_plus_age_sex"] = np.c_[r["identity_template"], r["age_sex"]]
        return r
    Rd, Rt = rep(dev, keep_d), rep(test, keep_t)
    sc = {k: AN.fit_and_score(Rd[k], y_d, dev["fold"][keep_d], Rt[k], PLAN["C_grid"],
                              pca=64 if k == "raw_median_beat" else None, seed=CFG["seed"]) for k in Rd}
    tab, aucs = AN.gains(y_t, sc, PLAN["n_boot"], PLAN["n_perm"], CFG["seed"])
    return tab, aucs, int(y_t.sum()), int((1 - y_t).sum())

def train_and_check(name, train_d, eval_d, seed=0, baseline_eer=None):
    """Train an identity model in runs/<name> (resumes / skips if finished), then apply the Phase 3 identity rule."""
    run = f"{PATHS['runs']}/{name}"
    fn = lambda m: {k: v for k, v in EV.verification(EV.embed(m, eval_d["signal"], WINDOW, DEVICE),
                                                       eval_d["patient_id"], eval_d["date"]).items() if k in ("eer", "rank1")}
    ecfg.freeze(CFG, run)
    TR.train_identity(CFG, train_d["signal"], train_d["patient_id"], run, DEVICE, eval_fn=fn, seed=seed)
    model, st = TR.load_best(CFG, run, DEVICE)
    r = EV.verification(EV.embed(model, eval_d["signal"], WINDOW, DEVICE), eval_d["patient_id"], eval_d["date"])
    ok = baseline_eer is None or r["eer"] <= PLAN["IDENTITY_MAX_RATIO"] * baseline_eer
    print(f"{name}: best epoch {st['epoch']} | EER {r['eer']:.3f} | rank-1 {r['rank1']:.3f} | identity check {'PASS' if ok else 'FAIL'}")
    return model, r, ok

print("Phase 4: GO | Device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "NO GPU")

### Cell 5.1: Lock the Phase 5 rules

In [ ]:
PLAN5 = {
    "written_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "seeds": [1, 2],
    "seed_rule": "the main claim holds if Delta_rand > 0 with CI excluding 0 for every seed (0, 1, 2)",
    "lead_II": "same pipeline on lead II; R-peak positions reused from lead I (same heartbeat timing)",
    "external_dataset": "CODE-15% (Zenodo 4916206, CC BY 4.0), file exams_part0",
    "external_identity_model": "the frozen PTB-XL identity model (id_main_s0), no retraining on CODE",
    "external_CD_equivalent": "positive = RBBB or LBBB or 1dAVb; negative = normal_ecg and none of the six labels",
    "external_records": "10 s exams only, lead DI, resampled 400 -> 500 Hz, one exam per patient (lowest exam_id)",
    "external_split": "patients split 70% dev (5 folds) / 30% test, stratified by label, seed 0",
}
p5 = f"{PATHS['gates']}/phase5_plan.json"
if os.path.isfile(p5):
    old = json.load(open(p5))
    changed = {k for k in PLAN5 if k != "written_at" and old.get(k) != PLAN5[k]}
    assert not changed, f"Phase 5 plan already written on {old['written_at']}; items differ: {changed}"
    PLAN5 = old
    print("Phase 5 plan already locked on", PLAN5["written_at"])
else:
    json.dump(PLAN5, open(p5, "w"), indent=2)
    print("Phase 5 plan locked:", p5)

### Cell 5.2: Two more random seeds

In [ ]:
seed_rows = []
s0 = {g["gain"]: g for g in gate3["gains"]}
seed_rows.append({"seed": 0, "eer": gate3["identity"]["eer"], **{f"{k}": s0[k]["estimate"] for k in s0},
                  "rand_ci_low": s0["Delta_rand"]["ci_low"], "rand_ci_high": s0["Delta_rand"]["ci_high"]})
base_eer = json.load(open(f"{PATHS['gates']}/gate_2.json"))["identity_baseline"]["eer"]
SEEDS_ID_OK = True
for seed in PLAN5["seeds"]:
    model, r, ok = train_and_check(f"id_main_s{seed}", D["id_train"], D["id_eval"], seed=seed, baseline_eer=base_eer)
    SEEDS_ID_OK &= ok
    tab, aucs, npos, nneg = leak_gains(model, D["probe_dev"], D["lockbox"], PRIMARY)
    seed_rows.append({"seed": seed, "eer": r["eer"], **tab["estimate"].to_dict(),
                      "rand_ci_low": tab.loc["Delta_rand", "ci_low"], "rand_ci_high": tab.loc["Delta_rand", "ci_high"]})
seeds = pd.DataFrame(seed_rows).set_index("seed")
seeds.to_csv(f"{OUT}/seeds_{PRIMARY}.csv")
print(seeds.round(4).to_string())
print("\nMean +- SD over 3 seeds:", {k: f"{seeds[k].mean():+.4f} +- {seeds[k].std():.4f}"
                                     for k in ["Delta_rand", "Delta_raw", "Delta_demo"]})
SEEDS_HOLD = bool((seeds["rand_ci_low"] > 0).all())
print("Seed rule (Delta_rand CI above 0 for every seed):", "PASS" if SEEDS_HOLD else "FAIL")

### Cell 5.3: Build the lead II cache

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
import wfdb
from scipy.signal import butter, sosfiltfilt

RAW = PATHS["raw"].rstrip("/") + "/"
LOCAL = CFG["data"]["local_dir"].rstrip("/") + "/"
CACHE2 = f"{PATHS['root']}/data/cache_leadII"
os.makedirs(CACHE2, exist_ok=True)
FS, N_SAMP = CFG["data"]["fs"], CFG["data"]["fs"] * CFG["data"]["record_seconds"]
b0, b1 = [int(round(ms * FS / 1000)) for ms in CFG["preprocess"]["beat_ms"]]
sos = butter(CFG["preprocess"]["filter_order"], CFG["preprocess"]["band_hz"], btype="band", fs=FS, output="sos")
db = pd.read_csv(RAW + "ptbxl_database.csv", index_col="ecg_id")

if all(os.path.isfile(f"{CACHE2}/{s}.npz") for s in D):
    print("Lead II cache already on Drive: skipped.")
else:
    stems = db["filename_hr"]
    todo = [s + e for s in stems for e in (".dat", ".hea") if not os.path.isfile(LOCAL + s + e)]
    print(f"Copying {len(todo):,} waveform files from Drive to the Colab disk (10-25 min, once)...")
    def copy_one(rel):
        os.makedirs(os.path.dirname(LOCAL + rel), exist_ok=True)
        shutil.copyfile(RAW + rel, LOCAL + rel)
    with ThreadPoolExecutor(max_workers=32) as pool:
        list(tqdm(pool.map(copy_one, todo), total=len(todo), mininterval=10))
    for s, d in D.items():
        sig = np.zeros((len(d["ecg_id"]), N_SAMP), np.float32)
        for i, e in enumerate(tqdm(d["ecg_id"], desc=f"lead II {s}", mininterval=10)):
            sig[i] = wfdb.rdrecord(LOCAL + stems.loc[e], channel_names=["II"]).p_signal[:, 0]
        filt = sosfiltfilt(sos, sig, axis=1).astype(np.float32)
        mb = np.full((len(sig), b1 - b0), np.nan, np.float32)
        for i in range(len(sig)):
            pk = d["rpeaks"][i][d["rpeaks"][i] >= 0]
            pk = pk[(pk + b0 >= 0) & (pk + b1 <= N_SAMP)]
            if len(pk):
                mb[i] = np.median(np.stack([filt[i, p + b0:p + b1] for p in pk]), axis=0)
        mb = np.where(np.isfinite(mb), mb, 0).astype(np.float32)
        sd = filt.std(1, keepdims=True)
        z = ((filt - filt.mean(1, keepdims=True)) / np.where(sd > 0, sd, 1)).astype(np.float16)
        np.savez(f"{CACHE2}/{s}.npz", **{k: v for k, v in d.items() if k not in ("signal", "median_beat", "date")},
                 signal=z, median_beat=mb)
        print(f"{s}: {len(sig):,} records -> {CACHE2}/{s}.npz")
D2 = {s: load_npz(f"{CACHE2}/{s}.npz") for s in D}

### Cell 5.4: Lead II identity model and leak

In [ ]:
base2 = EV.verification(EV.unit_rows(D2["id_eval"]["median_beat"]), D2["id_eval"]["patient_id"], D2["id_eval"]["date"])
print(f"Lead II identity baseline (median beat): EER {base2['eer']:.3f}")
model2, r2, LEAD2_ID_OK = train_and_check("id_lead2_s0", D2["id_train"], D2["id_eval"], seed=0, baseline_eer=base2["eer"])
tab2, aucs2, npos, nneg = leak_gains(model2, D2["probe_dev"], D2["lockbox"], PRIMARY)
tab2.to_csv(f"{OUT}/leadII_{PRIMARY}.csv")
print(f"\nLead II, {PRIMARY} vs NORM on the lockbox ({npos} vs {nneg}):\n" + tab2.round(4).to_string())

### Cell 5.5: Download and prepare CODE-15%

In [ ]:
import h5py
from scipy.signal import resample_poly
import neurokit2 as nk

CODE_DIR = "/content/code15"
CACHE3 = f"{PATHS['root']}/data/cache_code15"
os.makedirs(CODE_DIR, exist_ok=True); os.makedirs(CACHE3, exist_ok=True)
ZEN = "https://zenodo.org/records/4916206/files/"

if os.path.isfile(f"{CACHE3}/code_dev.npz") and os.path.isfile(f"{CACHE3}/code_test.npz"):
    print("CODE-15% cache already on Drive: skipped.")
else:
    for f in ["exams.csv", "exams_part0.zip"]:
        if not os.path.isfile(f"{CODE_DIR}/{f}") and not (f.endswith(".zip") and os.path.isfile(f"{CODE_DIR}/exams_part0.hdf5")):
            print(f"Downloading {f} from Zenodo (part0 is 2.7 GB, to the Colab disk only)...")
            rc = subprocess.run(["wget", "-c", "-q", "--show-progress", "--progress=dot:giga",
                                 "-O", f"{CODE_DIR}/{f}.part", ZEN + f + "?download=1"]).returncode
            assert rc == 0, f"Download of {f} stopped: re-run this cell to resume"
            os.replace(f"{CODE_DIR}/{f}.part", f"{CODE_DIR}/{f}")
    if not os.path.isfile(f"{CODE_DIR}/exams_part0.hdf5"):
        with zipfile.ZipFile(f"{CODE_DIR}/exams_part0.zip") as z:
            name = [n for n in z.namelist() if n.endswith(".hdf5")][0]
            with z.open(name) as src, open(f"{CODE_DIR}/exams_part0.hdf5", "wb") as out:
                shutil.copyfileobj(src, out, 1 << 24)
        os.remove(f"{CODE_DIR}/exams_part0.zip")

    ex = pd.read_csv(f"{CODE_DIR}/exams.csv")
    ex = ex[ex["trace_file"] == "exams_part0.hdf5"].set_index("exam_id")
    with h5py.File(f"{CODE_DIR}/exams_part0.hdf5", "r") as f:
        key_id = "exam_id" if "exam_id" in f else "id_exam"
        key_sig = "tracings" if "tracings" in f else "signal"
        ids = np.array(f[key_id])
        T = f[key_sig]
        lead = np.concatenate([T[i:i + 2000, :, 0] for i in range(0, T.shape[0], 2000)]).astype(np.float32)
    print(f"Read {len(ids):,} tracings, lead DI shape {lead.shape}")
    keep = (ids > 0) & np.isin(ids, ex.index)
    ids, lead = ids[keep], lead[keep]
    nz = np.abs(lead) > 0
    first = nz.argmax(1); last = lead.shape[1] - nz[:, ::-1].argmax(1)
    ten_s = (last - first) >= 3900
    ids, lead = ids[ten_s], lead[ten_s]
    meta = ex.loc[ids]
    flags = ["1dAVb", "RBBB", "LBBB", "SB", "AF", "ST"]
    cd_eq = meta[["RBBB", "LBBB", "1dAVb"]].astype(bool).any(axis=1).to_numpy()
    norm = (meta["normal_ecg"].astype(bool) & ~meta[flags].astype(bool).any(axis=1)).to_numpy()
    print(f"10 s exams: {len(ids):,} | CD-equivalent {cd_eq.sum():,} | normal {norm.sum():,}")

    sig500 = resample_poly(lead[:, 48:4048], 5, 4, axis=1).astype(np.float32)          # 4000 @400 Hz -> 5000 @500 Hz
    filt = sosfiltfilt(sos, sig500, axis=1).astype(np.float32)
    sd = filt.std(1, keepdims=True)
    z = ((filt - filt.mean(1, keepdims=True)) / np.where(sd > 0, sd, 1)).astype(np.float16)
    mb = np.full((len(z), b1 - b0), np.nan, np.float32)
    for i in tqdm(range(len(z)), desc="R-peaks", mininterval=10):
        try:
            pk = np.asarray(nk.ecg_peaks(filt[i], sampling_rate=FS)[1]["ECG_R_Peaks"], int)
            pk = pk[(pk + b0 >= 0) & (pk + b1 <= N_SAMP)]
            if len(pk) >= 3:
                mb[i] = np.median(np.stack([filt[i, p + b0:p + b1] for p in pk]), axis=0)
        except Exception:
            pass
    ok = np.isfinite(mb).all(1) & (sd[:, 0] > 0)
    labels = np.zeros((len(z), len(SUPER)), int)
    labels[:, SUPER.index("NORM")] = norm
    labels[:, SUPER.index(PRIMARY)] = cd_eq
    pid = meta["patient_id"].to_numpy()
    base = dict(ecg_id=ids, patient_id=pid, signal=z, median_beat=np.nan_to_num(mb), labels=labels,
                label_names=np.array(SUPER), age=meta["age"].to_numpy(float), sex=meta["is_male"].astype(float).to_numpy())

    # identity transfer set: patients with >= 2 good exams in this part (enroll = lowest exam_id)
    vc = pd.Series(pid[ok]).value_counts()
    rep_mask = ok & np.isin(pid, vc[vc >= 2].index)
    np.savez(f"{CACHE3}/code_identity.npz", **{k: (v[rep_mask] if k != "label_names" else v) for k, v in base.items()})

    # probe set: one exam per patient (lowest exam_id), only CD-equivalent or normal, split 70/30 stratified
    order = np.lexsort((ids, pid))
    firsts = order[np.r_[True, pid[order][1:] != pid[order][:-1]]]
    cand = firsts[ok[firsts] & (cd_eq[firsts] | norm[firsts])]
    rng = np.random.default_rng(0)
    test_idx, dev_idx = [], []
    for lab in (True, False):
        grp = cand[cd_eq[cand] == lab].copy(); rng.shuffle(grp)
        n_t = int(round(0.3 * len(grp))); test_idx += list(grp[:n_t]); dev_idx += list(grp[n_t:])
    dev_idx = np.array(dev_idx); test_idx = np.array(test_idx)
    fold = np.zeros(len(z), int); fold[dev_idx] = np.arange(len(dev_idx)) % 5
    for nm, ix in [("code_dev", dev_idx), ("code_test", test_idx)]:
        np.savez(f"{CACHE3}/{nm}.npz", **{k: (v[ix] if k != "label_names" else v) for k, v in base.items()}, fold=fold[ix])
        print(f"{nm}: {len(ix):,} patients | CD-equivalent {cd_eq[ix].sum():,} | normal {norm[ix].sum():,}")
    print(f"code_identity: {rep_mask.sum():,} exams from {(vc >= 2).sum():,} repeat patients")

def load_code(nm):
    d = load_npz(f"{CACHE3}/{nm}.npz", dates=False)
    d["date"] = d["ecg_id"].astype("datetime64[s]")          # CODE-15% has no dates: exam_id order stands in
    return d
C = {nm: load_code(nm) for nm in ["code_dev", "code_test", "code_identity"]}

### Cell 5.6: External test on CODE-15%

In [ ]:
id_model, _ = TR.load_best(CFG, f"{PATHS['runs']}/id_main_s0", DEVICE)
ci = C["code_identity"]
EXT = {}
if len(np.unique(ci["patient_id"])) >= 20:
    tr = EV.verification(EV.embed(id_model, ci["signal"], WINDOW, DEVICE), ci["patient_id"], ci["date"])
    bl = EV.verification(EV.unit_rows(ci["median_beat"]), ci["patient_id"], ci["date"])
    EXT["identity_transfer"] = {"eer": tr["eer"], "rank1": tr["rank1"], "baseline_eer": bl["eer"],
                                "n_patients": int(tr["n_enrolled"])}
    print(f"Identity transfer to CODE-15% (no retraining): EER {tr['eer']:.3f} vs median-beat {bl['eer']:.3f} "
          f"({tr['n_enrolled']} repeat patients)")
tab3, aucs3, npos, nneg = leak_gains(id_model, C["code_dev"], C["code_test"], PRIMARY)
tab3.to_csv(f"{OUT}/external_code15_{PRIMARY}.csv")
EXT["leak"] = tab3.round(4).reset_index().to_dict("records")
print(f"\nCODE-15% external test, CD-equivalent (RBBB/LBBB/1dAVb) vs normal ({npos} vs {nneg}):\n" + tab3.round(4).to_string())
if npos < 300:
    print("Note: fewer than 300 test positives; the CI will be wide. Adding exams_part1 would roughly double them.")

### Cell 5.7: Robustness summary and Phase 5 gate

In [ ]:
import matplotlib.pyplot as plt
rob = pd.DataFrame([
    {"setting": "PTB-XL lead I, seed 0 (main)", "est": seeds.loc[0, "Delta_rand"], "lo": seeds.loc[0, "rand_ci_low"], "hi": seeds.loc[0, "rand_ci_high"]},
    *[{"setting": f"PTB-XL lead I, seed {s}", "est": seeds.loc[s, "Delta_rand"], "lo": seeds.loc[s, "rand_ci_low"],
       "hi": seeds.loc[s, "rand_ci_high"]} for s in PLAN5["seeds"]],
    {"setting": "PTB-XL lead II", "est": tab2.loc["Delta_rand", "estimate"], "lo": tab2.loc["Delta_rand", "ci_low"], "hi": tab2.loc["Delta_rand", "ci_high"]},
    {"setting": "CODE-15% external (frozen model)", "est": tab3.loc["Delta_rand", "estimate"], "lo": tab3.loc["Delta_rand", "ci_low"], "hi": tab3.loc["Delta_rand", "ci_high"]},
]).set_index("setting")
rob.to_csv(f"{OUT}/robustness_Delta_rand.csv")
print(rob.round(4).to_string())

fig, ax = plt.subplots(figsize=(7.5, 3))
yy = np.arange(len(rob))[::-1]
ax.errorbar(rob["est"], yy, xerr=[rob["est"] - rob["lo"], rob["hi"] - rob["est"]], fmt="o", capsize=4, color="black")
ax.axvline(0, ls="--", c="grey"); ax.set_yticks(yy); ax.set_yticklabels(rob.index)
ax.set_xlabel("Delta_rand (95% CI)"); ax.set_title(f"Robustness of the {PRIMARY} leak")
plt.tight_layout(); plt.savefig(f"{PATHS['figures']}/figure4_robustness.png", dpi=200); plt.show()

checks = {"Seed runs pass identity check": SEEDS_ID_OK, "Lead II model passes identity check": LEAD2_ID_OK,
          "External CODE-15% test run": len(tab3) == 3}
for k, v in checks.items():
    print(f"  {'PASS' if v else 'FAIL'}  {k}")
holds = {"every seed": SEEDS_HOLD, "lead II": bool(tab2.loc["Delta_rand", "ci_low"] > 0),
         "external CODE-15%": bool(tab3.loc["Delta_rand", "ci_low"] > 0)}
for k, v in holds.items():
    print(f"  Leak (Delta_rand CI above 0) holds for {k}: {'YES' if v else 'NO'}")
json.dump({"phase": 5, "go": all(checks.values()), "checks": checks, "holds": holds,
           "seeds": seeds.round(4).reset_index().to_dict("records"), "lead_II": tab2.round(4).reset_index().to_dict("records"),
           "external": EXT, "environment": ecfg.environment()},
          open(f"{PATHS['gates']}/gate_5.json", "w"), indent=2, default=str)
print("GO: experiments complete. Next is Phase 6 (paper figures and writing)." if all(checks.values())
      else "NO-GO: fix the FAIL lines above.")